# Structure carries function — a like-to-like wiring rule across V1 / RL / AL

**Cohort.** 906 MICrONS neurons that are both proofread (`ax_clean`) and functionally matched, spanning V1, RL, AL across 13 scans. 11,822 directed synaptic connections.

**Research question.** *In this cohort, do monosynaptically connected neuron pairs share more similar visual responses than unconnected pairs — and does the effect survive once we control for spatial distance, brain area, layer, and cell type?*

This is the central claim of MICrONS Consortium et al., *"Functional connectomics reveals general wiring rule in mouse visual cortex"* (Nature, 2025). Here we test it on the local cohort using the canonical `C` (connectivity), `F` (stimulus responses), and `F_corr` (Pearson signal correlation) matrices.

**Hypotheses tested.**

| # | Claim | Where |
|---|---|---|
| H1 | Connected pairs have higher `F_corr` than unconnected pairs | §2 |
| H2 | Among connected pairs, larger synapse size → higher `F_corr` | §3 |
| H3 | Bidirectional > unidirectional > unconnected | §4 |
| H4 | H1 survives after controlling for soma–soma distance | §5 |
| H5 | H1 survives within same-area / same-layer / same-cell-type strata | §6 |
| H6 | Orientation similarity adds explanatory power on top of `F_corr` | §7 |
| H7 | Per-neuron hub strength correlates with mean inter-neuron `F_corr` | §8 |
| H8 | Reciprocal pairs and triangle motifs are over-represented vs a degree-preserving null | §9 |

**Plot palette.** Tableau-style categoricals for `none / uni / bi` and brain areas, **`RdBu_r`** for the signed `F_corr` heatmaps and scatters, **`mako`** for ordered-bin ramps. The §11 3D synthesis figure uses **the exact palette from `Dima/construct_matrices.ipynb`** — `coolwarm` edges with an `RdBu_r` colorbar over light-green / yellow / light-blue area cubes on a black background.


---
## §1 — Setup and data load

This section does the loading work once so the rest of the notebook is short.

**What happens here.** We (1) configure a Tableau-style matplotlib/seaborn theme, (2) build (or load from cache) the canonical 906-neuron cohort `matched`, the connectivity matrix `C` (906×906 sparse, weight = total synapse cleft volume), and the response matrix `F` (906×116 oracle stimuli), (3) compute the pairwise functional similarity `F_corr = corrcoef(F)`, and (4) assemble the all-pairs DataFrame `pairs` with one row per unordered neuron pair carrying its `f_corr`, `conn_type`, `dist_um`, area/layer/cell-type metadata, and orientation features. `pairs` is the workhorse for §2–§7.

**On caching.** Building `F` requires reading the ~20 GB functional HDF5; we cache `F`, `C`, and `matched` to `data/cache/` after the first run so reruns are seconds, not minutes.


In [ ]:
# 1.0  Imports + paths
from pathlib import Path
import json, time, warnings

import numpy as np
import pandas as pd
from scipy import sparse, stats
from scipy.stats import mannwhitneyu, spearmanr, pearsonr

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize, to_hex, ListedColormap

import seaborn as sns
import networkx as nx
import statsmodels.api as sm

warnings.filterwarnings('ignore', category=RuntimeWarning)

DATA_DIR   = Path('Data')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')
SYN_CSV    = DATA_DIR / '1718' / 'raw' / 'synapses_matched.csv'
FUNC_H5_CANDIDATES = list(dict.fromkeys([
    DATA_DIR / 'functional' / 'microns_functional.h5',
    Path('data') / 'functional' / 'microns_functional.h5',
    Path('Data') / 'functional' / 'microns_functional.h5',
]))
FUNC_H5    = next((p for p in FUNC_H5_CANDIDATES if p.exists()), FUNC_H5_CANDIDATES[0])
NODES_CSV  = DATA_DIR / 'exports' / 'G_906_nodes.csv'
EDGES_CSV  = DATA_DIR / 'exports' / 'G_906_edges.csv'
NODES_93   = DATA_DIR / 'exports' / 'G_93_nodes.csv'
EDGES_93   = DATA_DIR / 'exports' / 'G_93_edges.csv'
CACHE_DIR  = DATA_DIR / 'cache';        CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_FIG    = Path('outputs/figures');   OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TBL    = Path('outputs/tables');    OUT_TBL.mkdir(parents=True, exist_ok=True)

# Tableau-style theme
sns.set_theme(context='notebook', style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 150,
    'image.cmap': 'RdBu_r',
    'axes.titleweight': 'semibold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

# Connection categorical (matches compare_structural_functional.ipynb)
PAL_CONN = {'none': '#aab7c4', 'uni': '#4E79A7', 'bi': '#E15759'}
ORDER_CONN = ['none', 'uni', 'bi']

# Brain area categorical (matches visualize_structural.ipynb)
PAL_AREA = {'V1': '#4E79A7', 'RL': '#F28E2B', 'AL': '#59A14F'}

# Sequential ramp for ordered bins (quartiles, distance bins, hub continuous)
RAMP_SEQ = sns.color_palette('mako', as_cmap=True)
# Diverging map for signed F_corr (heatmaps, scatter color encoding)
RAMP_DIV = plt.get_cmap('RdBu_r')

print('Setup OK.  Repo root:', Path('.').resolve())


### 1.1 Build (or load) the canonical 906-neuron cohort

Cohort recipe is identical to `structural_network.ipynb` and `Dima/construct_matrices.ipynb`:
`fl.filter_neurons(units, proofread='ax_clean')` → `tuning='matched'` at materialization `version=1718`, deduped via `functional_data='best_only'`.

We cache the resulting DataFrame so reruns skip the network call.

In [ ]:
# 1.1  Build cohort (cached)
COHORT_PKL = CACHE_DIR / 'matched_906.pkl'
LOCAL_COHORT_CSV = Path('Leo/outputs/structural_network/all-matched_axon-clean/cohort_neurons.csv')

def normalize_matched(df):
    df = df.copy()
    if 'matrix_index' in df.columns:
        df = df.sort_values('matrix_index', kind='stable').reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)
    if 'unit_id' not in df.columns and 'functional_unit_id' in df.columns:
        df['unit_id'] = df['functional_unit_id']
    missing_tuning = [c for c in ['pref_ori', 'pref_dir', 'gOSI', 'gDSI'] if c not in df.columns]
    func_props_csv = DATA_DIR / '1718' / 'raw' / 'digital_twin_properties_bcm_coreg_v4.csv'
    if missing_tuning and func_props_csv.exists():
        func = pd.read_csv(func_props_csv)
        func_cols = ['session', 'scan_idx', 'unit_id', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI',
                     'cc_abs', 'cc_max', 'cc_norm', 'OSI', 'DSI']
        func = func[func_cols].copy()
        for c in ['session', 'scan_idx', 'unit_id']:
            func[c] = func[c].astype(int)
        merge_df = df.copy()
        for c in ['session', 'scan_idx', 'unit_id']:
            merge_df[c] = merge_df[c].astype(int)
        drop_cols = [c for c in func.columns if c in df.columns and c not in ['session', 'scan_idx', 'unit_id']]
        df = merge_df.drop(columns=drop_cols).merge(
            func,
            on=['session', 'scan_idx', 'unit_id'],
            how='left',
            validate='one_to_one',
        )
        if df[missing_tuning].isna().any().any():
            raise ValueError('Failed to merge tuning columns for every matched neuron')
    df.index.name = 'matrix_idx'
    return df

if COHORT_PKL.exists():
    matched = normalize_matched(pd.read_pickle(COHORT_PKL))
    if len(matched) != 906:
        print(f'ignoring stale cohort cache: {len(matched)} neurons')
        matched = None
    else:
        print(f'cohort loaded from cache: {len(matched)} neurons')
else:
    matched = None

if matched is None:
    if LOCAL_COHORT_CSV.exists():
        matched = normalize_matched(pd.read_csv(LOCAL_COHORT_CSV))
        matched.to_pickle(COHORT_PKL)
        print(f'cohort loaded from local csv and cached: {len(matched)} neurons')
    else:
        import microns_datacleaner as mic
        import microns_datacleaner.filters as fl
        cleaner = mic.MicronsDataCleaner(datadir=str(DATA_DIR) + '/', version=1718, download_policy='minimum')
        units, _ = cleaner.process_nucleus_data(functional_data='best_only')
        matched = normalize_matched(
            fl.filter_neurons(fl.filter_neurons(units, proofread='ax_clean'), tuning='matched')
        )
        matched.to_pickle(COHORT_PKL)
        print(f'cohort built and cached: {len(matched)} neurons')

assert len(matched) == 906, f'Expected 906 matched neurons, got {len(matched)}'
matched.to_pickle(COHORT_PKL)

print('areas: ', matched.brain_area.value_counts().to_dict())
print('layers:', matched.layer.value_counts().to_dict())


### 1.2 Build the connectivity matrix `C`

`C[i, j]` = total synaptic cleft `size` from neuron *i* (presynaptic) onto neuron *j* (postsynaptic), autapses removed. Source: `synapses_matched.csv` produced by `cleaner.download_synapse_data(matched_ids, matched_ids)`. Duplicate `(i, j)` entries collapse via `coo → csr`.

In [ ]:
# 1.2  Build C from the merged synapse CSV
N = len(matched)
pt_to_idx = pd.Series(matched.index.values, index=matched.pt_root_id)

syn = pd.read_csv(SYN_CSV)
syn = syn[syn.pre_pt_root_id != syn.post_pt_root_id]                           # drop autapses
syn = syn[syn.pre_pt_root_id.isin(pt_to_idx.index) & syn.post_pt_root_id.isin(pt_to_idx.index)]

rows = pt_to_idx.loc[syn.pre_pt_root_id].to_numpy()
cols = pt_to_idx.loc[syn.post_pt_root_id].to_numpy()
data = syn['size'].to_numpy(dtype=np.float32)

C = sparse.coo_matrix((data, (rows, cols)), shape=(N, N)).tocsr()
print(f'C: shape={C.shape}, nnz={C.nnz}, density={C.nnz / (N * N):.2e}')


### 1.3 Build the functional response matrix `F`

The 906 neurons span ~13 `(session, scan_idx)` scans. To make the columns of `F` mean the same thing across scans we align by `condition_hash` and keep only the **oracle conditions** (the intersection of conditions presented in every scan). For each scan we mean across trials of each oracle condition, then mean across time for each trial — yielding `F` of shape `(906, ≈116)` with `F[i, c]` = the average response of neuron *i* to oracle stimulus *c*.

Reading the 20 GB HDF5 takes a couple of minutes the first time, then is cached.

In [ ]:
# 1.3  Build F by aligning oracle stimulus conditions across scans (cached)
F_NPY        = CACHE_DIR / 'F_906.npy'
F_VALID_NPY  = CACHE_DIR / 'F_906_valid.npy'
F_COND_TXT   = CACHE_DIR / 'F_906_oracle_conditions.txt'

if F_NPY.exists() and F_VALID_NPY.exists():
    F = np.load(F_NPY)
    valid = np.load(F_VALID_NPY)
    print(f'F loaded from cache: {F.shape}, valid neurons: {valid.sum()}/{len(valid)}')
else:
    if not FUNC_H5.exists():
        searched = '\n'.join(f'  - {p}' for p in FUNC_H5_CANDIDATES)
        raise FileNotFoundError(
            'Missing MICrONS functional HDF5 needed to build F_906.npy.\n'
            'The 906-neuron cohort CSV is present, but the functional response matrix is not.\n'
            'Put microns_functional.h5 in Data/functional/ or data/functional/, then rerun this cell.\n'
            f'Searched:\n{searched}'
        )
    import h5py
    cohort = matched.assign(
        session_key=matched.session.astype(int).astype(str) + '_' + matched.scan_idx.astype(int).astype(str),
        unit_id_int=matched.unit_id.astype(int),
    )
    session_keys = sorted(cohort.session_key.unique())

    with h5py.File(FUNC_H5, 'r') as f:
        h5_sessions = set(f['sessions'].keys())
        usable = [sk for sk in session_keys if sk in h5_sessions]

        trials_cache, cond_sets = {}, []
        for sk in usable:
            trials = f[f'sessions/{sk}/trials']
            ch = [(tk, trials[tk].attrs['condition_hash']) for tk in trials.keys()]
            trials_cache[sk] = ch
            cond_sets.append({h for _, h in ch})

        common = sorted(set.intersection(*cond_sets))
        cond_to_col = {h: j for j, h in enumerate(common)}
        M = len(common)
        print(f'sessions used: {len(usable)} of {len(session_keys)}, oracle conditions: {M}')

        F = np.full((N, M), np.nan, dtype=np.float32)
        for sk in usable:
            sess = cohort[cohort.session_key == sk]
            h5_uids = f[f'sessions/{sk}/meta/unit_ids'][:]
            uid_to_row = {int(u): i for i, u in enumerate(h5_uids)}

            mi_list, h5_list = [], []
            for mi, uid in zip(sess.index, sess.unit_id_int):
                r = uid_to_row.get(uid)
                if r is not None:
                    mi_list.append(mi); h5_list.append(r)
            if not h5_list:
                continue
            mi_arr  = np.array(mi_list)
            h5_arr  = np.array(h5_list)

            per_cond = {}
            tg = f[f'sessions/{sk}/trials']
            for tk, ch in trials_cache[sk]:
                if ch not in cond_to_col:
                    continue
                resp = tg[tk]['responses'][:]                                # (n_units_in_sess, T)
                per_cond.setdefault(ch, []).append(resp[h5_arr].mean(axis=1))
            for ch, tms in per_cond.items():
                F[mi_arr, cond_to_col[ch]] = np.mean(np.stack(tms, axis=0), axis=0)

    valid = ~np.all(np.isnan(F), axis=1)
    np.save(F_NPY, F)
    np.save(F_VALID_NPY, valid)
    F_COND_TXT.write_text('\n'.join(common))
    print(f'F built: {F.shape}, valid neurons: {valid.sum()}/{len(valid)}, NaN frac: {np.isnan(F).mean():.3f}')


### 1.4 Functional similarity matrix `F_corr`

We use **Pearson correlation across the oracle-stimulus columns** (matches MICrONS 2025 and `compare_structural_functional.ipynb`). Diagonal = 1 by construction. Pairs involving a neuron whose session is missing from the HDF5 propagate NaN — we drop those pairs explicitly when building the pairs table below.

In [ ]:
# 1.4  F_corr (Pearson) + tiny preview heatmap (RdBu_r diverging — F_corr is signed)
F_corr = np.corrcoef(F)                                                        # (906, 906)
np.fill_diagonal(F_corr, 1.0)

print(f'F_corr: shape={F_corr.shape}')
mask_valid = valid.astype(bool)
off = ~np.eye(N, dtype=bool)
finite = np.isfinite(F_corr) & off
print(f'finite off-diag entries: {finite.sum():,} / {off.sum():,}')
print(f'mean F_corr (finite off-diag): {F_corr[finite].mean():+.4f}')

# Quick look at the matrix sorted by brain area so any area-block structure pops out visually
order = np.argsort(matched.brain_area.map({'V1': 0, 'RL': 1, 'AL': 2}).values, kind='stable')
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(F_corr[np.ix_(order, order)], cmap='RdBu_r', vmin=-0.6, vmax=0.6,
               aspect='equal', interpolation='nearest')
boundaries = np.cumsum(matched.brain_area.value_counts().reindex(['V1', 'RL', 'AL']).values)
for b in boundaries[:-1]:
    ax.axhline(b - 0.5, color='black', lw=0.7, alpha=0.5)
    ax.axvline(b - 0.5, color='black', lw=0.7, alpha=0.5)
ax.set(title=f'Functional similarity F_corr  (sorted V1 → RL → AL, n={N})',
       xlabel='neuron j', ylabel='neuron i')
fig.colorbar(im, ax=ax, label='Pearson correlation', shrink=0.85)
plt.tight_layout(); plt.show()


### 1.5 The all-pairs table `pairs`

One row per unordered pair `(i, j)` with `i < j`, restricted to neurons where `F_corr` is finite (i.e., both neurons have valid functional responses). Columns:

- `f_corr` — Pearson signal correlation
- `conn_type` ∈ {`none`, `uni`, `bi`} — derived from `C` (presence of `i→j` and/or `j→i` synapses)
- `syn_size_max` — max(C[i,j], C[j,i]); 0 for unconnected pairs
- `dist_um` — Euclidean soma–soma distance in µm
- `same_area`, `same_layer`, `same_celltype` — boolean
- `ori_sim` — `cos(2·Δθ)` where θ is `pref_ori`; NaN if either neuron lacks tuning
- `gOSI_min` — `min(gOSI_i, gOSI_j)` for filtering well-tuned subsets in §7

This is the workhorse for §2–§7.

In [ ]:
# 1.5  Build the all-pairs DataFrame
xyz = matched[['pt_position_x', 'pt_position_y', 'pt_position_z']].to_numpy()
ori = np.deg2rad(matched['pref_ori'].to_numpy())
gOSI = matched['gOSI'].to_numpy()
area = matched['brain_area'].to_numpy()
layer = matched['layer'].to_numpy()
ctype = matched['cell_type'].to_numpy()

C_dense = C.toarray()
A_und = (C_dense > 0) | (C_dense.T > 0)
np.fill_diagonal(A_und, False)
bidir  = (C_dense > 0) & (C_dense.T > 0)

iu, ju = np.triu_indices(N, k=1)
finite_mask = np.isfinite(F_corr[iu, ju])
iu, ju = iu[finite_mask], ju[finite_mask]

pairs = pd.DataFrame({
    'i': iu, 'j': ju,
    'f_corr':       F_corr[iu, ju],
    'syn_ij':       C_dense[iu, ju],
    'syn_ji':       C_dense[ju, iu],
})
pairs['syn_size_max'] = np.maximum(pairs.syn_ij, pairs.syn_ji)
pairs['conn_type'] = np.where(
    bidir[iu, ju], 'bi',
    np.where(A_und[iu, ju], 'uni', 'none'),
)
pairs['connected']    = pairs.conn_type != 'none'
pairs['dist_um']      = np.linalg.norm(xyz[iu] - xyz[ju], axis=1)
pairs['same_area']    = area[iu] == area[ju]
pairs['same_layer']   = layer[iu] == layer[ju]
pairs['same_celltype']= ctype[iu] == ctype[ju]

dtheta = ori[iu] - ori[ju]
pairs['ori_sim'] = np.cos(2.0 * dtheta)
pairs['gOSI_min'] = np.minimum(gOSI[iu], gOSI[ju])

print(f'pairs: {len(pairs):,}   conn_type counts: {pairs.conn_type.value_counts().to_dict()}')
pairs.head()


---
## §2 — H1: Connected vs unconnected pairs (the headline test)

**Question.** Is `mean F_corr | connected > mean F_corr | unconnected`?

**What we report.**
- A side-by-side **violin** of `f_corr` for `none / uni / bi`, tinted with the categorical Tableau palette.
- A **one-sided Mann–Whitney U** (`connected > unconnected`).
- A **paired bootstrap 95 % CI** for the mean difference (10 k resamples).
- **Cohen's d** as a complementary effect size.

A positive Δmean with a CI excluding 0 and U statistic well into the right tail is what "like-to-like wiring" looks like in this cohort.

In [ ]:
# 2.0  H1 — connected vs unconnected
def boot_mean_diff(a, b, n_boot=10_000, seed=0):
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        diffs[k] = rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
    return diffs.mean(), np.percentile(diffs, [2.5, 97.5])

def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp

f_none = pairs.loc[pairs.conn_type == 'none', 'f_corr'].to_numpy()
f_uni  = pairs.loc[pairs.conn_type == 'uni',  'f_corr'].to_numpy()
f_bi   = pairs.loc[pairs.conn_type == 'bi',   'f_corr'].to_numpy()
f_conn = pairs.loc[pairs.connected, 'f_corr'].to_numpy()

U, p = mannwhitneyu(f_conn, f_none, alternative='greater')
md_, ci = boot_mean_diff(f_conn, f_none)
d = cohens_d(f_conn, f_none)
print(f'connected (n={len(f_conn):,}) vs unconnected (n={len(f_none):,})')
print(f'  Δmean = {md_:+.4f}   95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]')
print(f'  Mann–Whitney U = {U:.3e}   one-sided p = {p:.3e}')
print(f'  Cohen\'s d = {d:+.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={'width_ratios': [1.2, 1]})

ax = axes[0]
data = [f_none, f_uni, f_bi]
parts = ax.violinplot(data, showmeans=False, showmedians=True, widths=0.85)
for k, pc in enumerate(parts['bodies']):
    pc.set_facecolor(PAL_CONN[ORDER_CONN[k]])
    pc.set_edgecolor('white'); pc.set_alpha(0.85)
for key in ('cmedians', 'cbars', 'cmins', 'cmaxes'):
    parts[key].set_color('0.25'); parts[key].set_linewidth(1.0)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([f'none\nn={len(f_none):,}',
                    f'uni\nn={len(f_uni):,}',
                    f'bi\nn={len(f_bi):,}'])
ax.set_ylabel('Pearson signal correlation  $F_{corr}$')
ax.set_title('H1 — connected pairs are more functionally similar')
ax.axhline(0, color='0.4', lw=0.7, ls=':')

ax = axes[1]
groups = ['none', 'uni', 'bi']
means, los, his = [], [], []
rng = np.random.default_rng(1)
for g in groups:
    x = pairs.loc[pairs.conn_type == g, 'f_corr'].to_numpy()
    bm = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(2000)])
    means.append(x.mean()); los.append(np.percentile(bm, 2.5)); his.append(np.percentile(bm, 97.5))
xpos = np.arange(3)
colors = [PAL_CONN[g] for g in groups]
ax.bar(xpos, means, yerr=[np.array(means) - np.array(los), np.array(his) - np.array(means)],
       color=colors, edgecolor='white', linewidth=0.8, capsize=5)
ax.set_xticks(xpos); ax.set_xticklabels(groups)
ax.set_ylabel('mean $F_{corr}$  (95% CI)')
ax.set_title(f'Δmean (conn − none) = {md_:+.4f}, p = {p:.1e}')
plt.tight_layout(); plt.show()

H1_RESULT = {'test': 'H1 connected > unconnected',
             'n_conn': int(len(f_conn)), 'n_none': int(len(f_none)),
             'mean_diff': float(md_), 'ci_lo': float(ci[0]), 'ci_hi': float(ci[1]),
             'cohens_d': float(d), 'U': float(U), 'p_one_sided': float(p)}


---
## §3 — H2: synapse strength gradient

**Question.** Among connected pairs, do larger synapses (in cleft volume) predict higher `F_corr`?

We use **Spearman** as the primary statistic because synapse-size is heavy-tailed; Pearson on `log10(size)` is reported as a secondary check.

A scatter on the left shows every connected pair coloured by `f_corr` (`RdBu_r`, since `f_corr` is signed); a binned mean ± SEM bar plot on the right uses the `mako` ordered ramp.

In [ ]:
# 3.0  H2 — synapse strength gradient (connected pairs only)
conn = pairs[pairs.connected].copy()
conn['log10_size'] = np.log10(conn.syn_size_max)

rho_s, p_s = spearmanr(conn.syn_size_max, conn.f_corr)
rho_p, p_p = pearsonr(conn.log10_size, conn.f_corr)
print(f'Spearman(size, F_corr)         = {rho_s:+.3f}   p = {p_s:.2e}   n = {len(conn):,}')
print(f'Pearson(log10 size, F_corr)    = {rho_p:+.3f}   p = {p_p:.2e}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
fc_lim = max(abs(np.percentile(conn.f_corr, 1)), abs(np.percentile(conn.f_corr, 99)))
sc = ax.scatter(conn.log10_size, conn.f_corr, c=conn.f_corr, cmap='RdBu_r',
                vmin=-fc_lim, vmax=+fc_lim, s=10, alpha=0.6, linewidths=0)
m, b = np.polyfit(conn.log10_size, conn.f_corr, 1)
xs = np.linspace(conn.log10_size.min(), conn.log10_size.max(), 50)
ax.plot(xs, m * xs + b, color='0.15', lw=1.4, ls='--', label=f'OLS slope = {m:+.3f}')
ax.set(xlabel='log10 max synapse size  (cleft volume)',
       ylabel='$F_{corr}$',
       title=f'H2 scatter — Spearman ρ = {rho_s:+.3f}, p = {p_s:.1e}')
ax.legend(loc='lower right')
fig.colorbar(sc, ax=ax, label='$F_{corr}$', shrink=0.85)

ax = axes[1]
conn['quartile'] = pd.qcut(conn.syn_size_max, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
g = conn.groupby('quartile').f_corr.agg(['mean', 'sem', 'count'])
colors = RAMP_SEQ(np.linspace(0.25, 0.85, 4))
ax.bar(np.arange(4), g['mean'], yerr=g['sem'], color=colors, edgecolor='white', capsize=4)
for k, (mu, n) in enumerate(zip(g['mean'], g['count'])):
    ax.text(k, mu + 0.005, f'n={n}', ha='center', va='bottom', fontsize=9, color='0.25')
ax.set_xticks(np.arange(4)); ax.set_xticklabels(g.index)
ax.set(ylabel='mean $F_{corr}$ (± SEM)',
       xlabel='synapse-size quartile',
       title='H2 quartiles — graded by connection strength')
plt.tight_layout(); plt.show()

H2_RESULT = {'test': 'H2 synapse-size gradient', 'spearman_rho': float(rho_s),
             'p_spearman': float(p_s), 'n_connected': int(len(conn))}


---
## §4 — H3: reciprocity premium (`bi > uni > none`)

**Question.** Are bidirectional pairs even more functionally similar than unidirectional ones?

Same Tableau categorical (gray / blue / red) as the H1 panel. We bootstrap a 95 % CI on each group's mean and run pairwise one-sided Mann–Whitney tests. **Caveat:** in the 906-cohort the bidirectional sample is small; when `n_bi < 30` only the `uni > none` test is well-powered.

In [ ]:
# 4.0  H3 — reciprocity premium
def boot_mean_ci(x, n=4000, seed=0):
    rng = np.random.default_rng(seed)
    bm = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(n)])
    return x.mean(), np.percentile(bm, 2.5), np.percentile(bm, 97.5)

groups = {'none': f_none, 'uni': f_uni, 'bi': f_bi}
mu, lo, hi = [], [], []
for g in ORDER_CONN:
    m_, l_, h_ = boot_mean_ci(groups[g])
    mu.append(m_); lo.append(l_); hi.append(h_)
    print(f'{g:>4s}  n={len(groups[g]):>7,d}   mean F_corr = {m_:+.4f}   95% CI [{l_:+.4f}, {h_:+.4f}]')

tests = [('bi vs none', f_bi, f_none),
         ('bi vs uni',  f_bi, f_uni),
         ('uni vs none', f_uni, f_none)]
for name, a, b in tests:
    U_, p_ = mannwhitneyu(a, b, alternative='greater')
    print(f'  {name:>11s}  one-sided p = {p_:.3e}')

fig, ax = plt.subplots(figsize=(7, 4.5))
xs = np.arange(3)
ax.bar(xs, mu, yerr=[np.array(mu) - np.array(lo), np.array(hi) - np.array(mu)],
       color=[PAL_CONN[g] for g in ORDER_CONN], edgecolor='white', capsize=6, linewidth=0.8)
for k, (g, m_) in enumerate(zip(ORDER_CONN, mu)):
    ax.text(k, m_ + 0.005, f'n={len(groups[g]):,}', ha='center', va='bottom', fontsize=10, color='0.25')
ax.set_xticks(xs); ax.set_xticklabels(ORDER_CONN)
ax.set_ylabel('mean $F_{corr}$  (95% CI)')
ax.set_title('H3 — reciprocity premium')
plt.tight_layout(); plt.show()

H3_RESULT = {'test': 'H3 reciprocity', 'n_bi': int(len(f_bi)), 'n_uni': int(len(f_uni)),
             'n_none': int(len(f_none)), 'mean_bi': float(np.mean(f_bi)),
             'mean_uni': float(np.mean(f_uni)), 'mean_none': float(np.mean(f_none))}


---
## §5 — H4: distance control (the hardest confound)

Both `P(connected | dist)` and `mean F_corr | dist` decay with soma–soma distance, so a naive comparison double-counts that. We run **four** distance-aware tests:

1. Plot `P(connected | dist)` and `mean F_corr | dist` on the same axis (twin y) over distance bins — the two lines use the connected/none Tableau colors so it's visually obvious which curve belongs to which axis.
2. **Distance-matched control.** For each connected pair, sample an unconnected pair with the closest matching `dist_um` (greedy nearest neighbour, without replacement). Recompute the H1 test on this matched set.
3. **Logistic regression.** `connected ~ f_corr + dist + dist²`. The partial coefficient on `f_corr` is the question; HC0 robust SE because pairs are not iid.
4. **Within-bin tests.** Five distance quintiles, an MW(connected vs unconnected) inside each, FDR-corrected — a forest plot summarising whether the effect survives at every spatial scale, with the `mako` ramp encoding bin position.

In [ ]:
# 5.0  H4.1 — twin-axis distance profile of P(conn) and F_corr
bins = np.linspace(pairs.dist_um.min(), pairs.dist_um.max(), 25)
mid = 0.5 * (bins[:-1] + bins[1:])
binned = pairs.groupby(pd.cut(pairs.dist_um, bins, include_lowest=True), observed=False)
p_conn = binned.connected.mean().values
mean_fc = binned.f_corr.mean().values

c_struct = PAL_CONN['uni']      # blue — structure
c_func   = PAL_CONN['bi']       # red — function

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax2 = ax.twinx()
ax.plot(mid, p_conn, color=c_struct, lw=2.2, label='P(connected | dist)')
ax2.plot(mid, mean_fc, color=c_func, lw=2.2, ls='--', label='mean $F_{corr}$ | dist')
ax.set_xlabel('soma–soma distance  (µm)')
ax.set_ylabel('P(connected)', color=c_struct)
ax2.set_ylabel('mean $F_{corr}$', color=c_func)
ax.tick_params(axis='y', colors=c_struct)
ax2.tick_params(axis='y', colors=c_func)
ax.grid(False); ax2.grid(False)
ax.set_title('H4.1 — both quantities decay with distance')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.85))
plt.tight_layout(); plt.show()


In [ ]:
# 5.1  H4.2 — distance-matched H1 retest
rng = np.random.default_rng(0)
conn_idx   = pairs.index[pairs.connected].to_numpy()
unconn_idx = pairs.index[~pairs.connected].to_numpy()

unconn_sorted = unconn_idx[np.argsort(pairs.loc[unconn_idx, 'dist_um'].to_numpy())]
unconn_dist_sorted = pairs.loc[unconn_sorted, 'dist_um'].to_numpy()
matched_unconn = []
used = np.zeros(len(unconn_sorted), dtype=bool)
shuf = rng.permutation(conn_idx)
for ci in shuf:
    d = pairs.loc[ci, 'dist_um']
    pos = np.searchsorted(unconn_dist_sorted, d)
    best = None; best_diff = np.inf
    for off in range(0, 50):
        for k in (pos - off, pos + off):
            if 0 <= k < len(unconn_sorted) and not used[k]:
                diff = abs(unconn_dist_sorted[k] - d)
                if diff < best_diff:
                    best_diff = diff; best = k
        if best is not None and best_diff < 2.0:
            break
    if best is not None:
        used[best] = True
        matched_unconn.append(unconn_sorted[best])

matched_unconn = np.array(matched_unconn)
fc_match = pairs.loc[matched_unconn, 'f_corr'].to_numpy()
print(f'matched pairs: {len(matched_unconn):,} (out of {len(conn_idx):,} connected)')

U, p = mannwhitneyu(f_conn, fc_match, alternative='greater')
md_match, ci_match = boot_mean_diff(f_conn, fc_match)
print(f'distance-matched Δmean = {md_match:+.4f}   95% CI [{ci_match[0]:+.4f}, {ci_match[1]:+.4f}]   p = {p:.3e}')

H4_MATCH = {'test': 'H4 distance-matched', 'mean_diff': float(md_match),
            'ci_lo': float(ci_match[0]), 'ci_hi': float(ci_match[1]), 'p_one_sided': float(p)}

fig, ax = plt.subplots(figsize=(8.5, 4.6))
for label, x, c in [
    ('unconnected (all)',          f_none,   PAL_CONN['none']),
    ('unconnected (dist-matched)', fc_match, '#7F9BAE'),
    ('connected',                  f_conn,   PAL_CONN['uni']),
]:
    ax.hist(x, bins=80, density=True, histtype='step', linewidth=1.8, color=c, label=f'{label} (n={len(x):,})')
ax.axvline(0, color='0.4', lw=0.6, ls=':')
ax.set(xlabel='$F_{corr}$', ylabel='density',
       title=f'H4.2 — distance-matched: Δmean = {md_match:+.4f}, p = {p:.1e}')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 5.2  H4.3 — logistic regression connected ~ f_corr + dist + dist^2  (HC0 SEs)
X = pairs[['f_corr', 'dist_um']].to_numpy()
X = np.column_stack([X, X[:, 1] ** 2])
X = sm.add_constant(X)
y = pairs.connected.to_numpy().astype(int)

logit = sm.Logit(y, X).fit(disp=False, cov_type='HC0')
print(logit.summary().tables[1])

names = ['const', 'f_corr', 'dist', 'dist^2']
beta_fc, se_fc, p_fc = logit.params[1], logit.bse[1], logit.pvalues[1]
print(f'\npartial coefficient on f_corr: β = {beta_fc:+.3f}  ± {se_fc:.3f}  (p = {p_fc:.2e})')
H4_LOGIT = {'test': 'H4 logistic (f_corr | dist)',
            'beta_fcorr': float(beta_fc), 'se_fcorr': float(se_fc), 'p_fcorr': float(p_fc)}


In [ ]:
# 5.3  H4.4 — within-distance-bin MW + forest plot (BH-FDR corrected)
qs = pairs['dist_um'].quantile(np.linspace(0, 1, 6)).to_numpy()
labels, ds, los, his, ps = [], [], [], [], []
for k in range(5):
    sel = (pairs.dist_um >= qs[k]) & (pairs.dist_um < qs[k + 1] if k < 4 else pairs.dist_um <= qs[k + 1])
    sub = pairs[sel]
    a = sub.loc[sub.connected, 'f_corr'].to_numpy()
    b = sub.loc[~sub.connected, 'f_corr'].to_numpy()
    if len(a) < 5 or len(b) < 5:
        continue
    md_, ci_ = boot_mean_diff(a, b, n_boot=4000, seed=k)
    _, p_ = mannwhitneyu(a, b, alternative='greater')
    labels.append(f'Q{k+1}: {qs[k]:.0f}–{qs[k+1]:.0f} µm\n(n_c={len(a)}, n_u={len(b)})')
    ds.append(md_); los.append(ci_[0]); his.append(ci_[1]); ps.append(p_)

ranked = np.argsort(ps)
m = len(ps)
adj = np.empty(m); cur_min = 1.0
for r in range(m - 1, -1, -1):
    idx = ranked[r]
    cur_min = min(cur_min, ps[idx] * m / (r + 1))
    adj[idx] = cur_min

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ypos = np.arange(len(ds))[::-1]
colors_q = RAMP_SEQ(np.linspace(0.25, 0.85, len(ds)))
for y_, d_, l_, h_, c_, p_, q_ in zip(ypos, ds, los, his, colors_q, ps, adj):
    ax.errorbar(d_, y_, xerr=[[d_ - l_], [h_ - d_]], fmt='o', color=c_,
                markersize=8, capsize=4, lw=1.5)
    ax.text(h_ + 0.005, y_, f'p={p_:.1e}  q={q_:.1e}', va='center', fontsize=8.5, color='0.3')
ax.axvline(0, color='0.4', lw=0.7, ls=':')
ax.set_yticks(ypos); ax.set_yticklabels(labels)
ax.set_xlabel('Δ mean $F_{corr}$  (connected − unconnected, 95% CI)')
ax.set_title('H4.4 — within-distance-bin tests (BH-FDR corrected)')
plt.tight_layout(); plt.show()


---
## §6 — H5: composition control (area / layer / cell type)

Same-area / same-layer / same-cell-type pairs are both more likely to be connected and more functionally similar — so part of H1 could just be "the cohort has compositional structure". We:

1. Stratify by `same_area`, `same_layer`, `same_celltype` and rerun the H1 test inside each stratum (forest plot uses the `mako` ramp).
2. Fit a **joint logistic regression** `connected ~ f_corr + dist + same_area + same_layer + same_celltype`. Standardised coefficients tell us how much explanatory power each feature carries when the others are held constant.

If `f_corr` has a positive coefficient inside this model, like-to-like is robust to compositional confounding.

In [ ]:
# 6.0  H5 — stratified H1 + joint logistic regression
strat = {}
for col in ['same_area', 'same_layer', 'same_celltype']:
    rows = []
    for v in [True, False]:
        sub = pairs[pairs[col] == v]
        a = sub.loc[sub.connected, 'f_corr'].to_numpy()
        b = sub.loc[~sub.connected, 'f_corr'].to_numpy()
        md_, ci_ = boot_mean_diff(a, b, n_boot=2000, seed=hash((col, v)) & 0xffff)
        _, p_ = mannwhitneyu(a, b, alternative='greater')
        rows.append({'stratum': f'{col}={v}',
                     'n_conn': len(a), 'n_unconn': len(b),
                     'mean_diff': md_, 'ci_lo': ci_[0], 'ci_hi': ci_[1], 'p': p_})
    strat[col] = pd.DataFrame(rows)
strat_df = pd.concat(strat.values(), ignore_index=True)
print(strat_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ypos = np.arange(len(strat_df))[::-1]
cols = RAMP_SEQ(np.linspace(0.25, 0.85, len(strat_df)))
for y_, row, c_ in zip(ypos, strat_df.itertuples(), cols):
    ax.errorbar(row.mean_diff, y_, xerr=[[row.mean_diff - row.ci_lo], [row.ci_hi - row.mean_diff]],
                fmt='o', color=c_, markersize=8, capsize=4, lw=1.5)
    ax.text(row.ci_hi + 0.003, y_, f'p={row.p:.1e}', va='center', fontsize=8.5, color='0.3')
ax.axvline(0, color='0.4', lw=0.7, ls=':')
ax.set_yticks(ypos); ax.set_yticklabels(strat_df.stratum)
ax.set_xlabel('Δ mean $F_{corr}$  (connected − unconnected)')
ax.set_title('H5 — stratified H1 (within / across area, layer, cell type)')
plt.tight_layout(); plt.show()

def zscore(x):
    return (x - x.mean()) / x.std(ddof=0)
Xj = pd.DataFrame({
    'f_corr':         zscore(pairs.f_corr.values),
    'dist_um':        zscore(pairs.dist_um.values),
    'same_area':      pairs.same_area.astype(int).values,
    'same_layer':     pairs.same_layer.astype(int).values,
    'same_celltype':  pairs.same_celltype.astype(int).values,
})
Xj = sm.add_constant(Xj)
yj = pairs.connected.astype(int).values
joint = sm.Logit(yj, Xj).fit(disp=False, cov_type='HC0')
print('\nJoint logistic (standardised continuous, raw 0/1 categoricals, HC0 SE):')
print(joint.summary().tables[1])

H5_RESULT = {'test': 'H5 joint logistic',
             'beta_fcorr': float(joint.params['f_corr']),
             'se_fcorr':   float(joint.bse['f_corr']),
             'p_fcorr':    float(joint.pvalues['f_corr'])}


---
## §7 — H6: orientation tuning subset

Orientation similarity `cos(2·Δθ)` is the classic like-to-like feature (Ko et al. 2011). Restricted to **well-tuned** pairs (`gOSI_min > 0.1` — conventional cortical cutoff), we ask:

1. Is `ori_sim` higher for connected than unconnected pairs?
2. In a logistic regression `connected ~ f_corr + ori_sim + dist`, which carries more weight? Wang et al. 2025 found `f_corr` (signal correlation) wins; we report whether our cohort agrees, with a bar of standardised β coefficients on a `mako` ramp.

In [ ]:
# 7.0  H6 — orientation subset
sub = pairs.dropna(subset=['ori_sim']).copy()
sub = sub[sub.gOSI_min > 0.10]
print(f'well-tuned pairs (gOSI_min > 0.1): {len(sub):,}')

a_o = sub.loc[sub.connected, 'ori_sim'].to_numpy()
b_o = sub.loc[~sub.connected, 'ori_sim'].to_numpy()
U_o, p_o = mannwhitneyu(a_o, b_o, alternative='greater')
md_o, ci_o = boot_mean_diff(a_o, b_o, n_boot=4000)
print(f'ori_sim  Δmean = {md_o:+.4f}   95% CI [{ci_o[0]:+.4f}, {ci_o[1]:+.4f}]   p = {p_o:.2e}')

Xs = pd.DataFrame({
    'f_corr':  (sub.f_corr   - sub.f_corr.mean())   / sub.f_corr.std(ddof=0),
    'ori_sim': (sub.ori_sim  - sub.ori_sim.mean())  / sub.ori_sim.std(ddof=0),
    'dist_um': (sub.dist_um  - sub.dist_um.mean())  / sub.dist_um.std(ddof=0),
})
Xs = sm.add_constant(Xs)
ys = sub.connected.astype(int).values
m6 = sm.Logit(ys, Xs).fit(disp=False, cov_type='HC0')
print('\nJoint logistic (standardised, HC0 SE):')
print(m6.summary().tables[1])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
ax.hist(b_o, bins=40, density=True, histtype='step', lw=1.8, color=PAL_CONN['none'],
        label=f'unconnected (n={len(b_o):,})')
ax.hist(a_o, bins=40, density=True, histtype='step', lw=1.8, color=PAL_CONN['uni'],
        label=f'connected (n={len(a_o):,})')
ax.set(xlabel='orientation similarity  $\\cos(2\\Delta\\theta)$', ylabel='density',
       title=f'H6 — orientation similarity (Δmean = {md_o:+.4f}, p = {p_o:.1e})')
ax.legend()

ax = axes[1]
betas = [m6.params['f_corr'], m6.params['ori_sim'], m6.params['dist_um']]
ses   = [m6.bse['f_corr'],    m6.bse['ori_sim'],    m6.bse['dist_um']]
ax.bar(np.arange(3), betas, yerr=1.96 * np.array(ses),
       color=RAMP_SEQ(np.linspace(0.30, 0.85, 3)), edgecolor='white', capsize=5)
ax.axhline(0, color='0.4', lw=0.7, ls=':')
ax.set_xticks(np.arange(3)); ax.set_xticklabels(['f_corr', 'ori_sim', 'dist'])
ax.set_ylabel('standardised logistic β  (95% CI)')
ax.set_title('Which feature carries weight given the others?')
plt.tight_layout(); plt.show()

H6_RESULT = {'test': 'H6 orientation', 'mean_diff_ori_sim': float(md_o),
             'p_ori_sim': float(p_o),
             'beta_fcorr_joint': float(m6.params['f_corr']),
             'beta_orisim_joint': float(m6.params['ori_sim']),
             'n_pairs': int(len(sub))}


---
## §8 — H7: hub coupling

Per neuron: total degree, total synapse strength, and mean `F_corr` with the rest of the cohort. Are structural hubs also functional hubs? We report Spearman correlations and a **partial** Spearman that controls for the neuron's mean inter-neuron distance — to make sure "central neurons are physically central" isn't doing all the work.

Scatter colour encodes per-neuron `mean_fcorr` with `RdBu_r` so the few neurons whose mean correlation is slightly negative pop visually.

In [ ]:
# 8.0  H7 — hub coupling
F_corr_clean = F_corr.copy()
np.fill_diagonal(F_corr_clean, np.nan)

mean_fcorr  = np.nanmean(F_corr_clean, axis=1)
total_deg   = np.asarray((C != 0).sum(axis=1)).ravel() + np.asarray((C != 0).sum(axis=0)).ravel()
total_str   = np.asarray(C.sum(axis=1)).ravel() + np.asarray(C.sum(axis=0)).ravel()
mean_dist   = np.array([np.linalg.norm(xyz - xyz[i], axis=1).sum() / (N - 1) for i in range(N)])

vmask = valid & np.isfinite(mean_fcorr)
df_h = pd.DataFrame({
    'neuron': np.arange(N),
    'mean_fcorr':  mean_fcorr,
    'total_deg':   total_deg.astype(float),
    'total_str':   total_str,
    'mean_dist':   mean_dist,
    'area':        matched.brain_area.values,
})[vmask].copy()

def partial_spearman(x, y, z):
    rx = stats.rankdata(x); ry = stats.rankdata(y); rz = stats.rankdata(z)
    A = np.column_stack([np.ones_like(rz), rz])
    bx = np.linalg.lstsq(A, rx, rcond=None)[0]; ex = rx - A @ bx
    by = np.linalg.lstsq(A, ry, rcond=None)[0]; ey = ry - A @ by
    return pearsonr(ex, ey)

rho_d, p_d = spearmanr(df_h.total_deg, df_h.mean_fcorr)
rho_s, p_s = spearmanr(df_h.total_str, df_h.mean_fcorr)
prho_d, pp_d = partial_spearman(df_h.total_deg.values, df_h.mean_fcorr.values, df_h.mean_dist.values)
prho_s, pp_s = partial_spearman(df_h.total_str.values, df_h.mean_fcorr.values, df_h.mean_dist.values)
print(f'Spearman(total_deg, mean F_corr)         = {rho_d:+.3f}  p = {p_d:.2e}')
print(f'partial   (total_deg, mean F_corr | dist)= {prho_d:+.3f}  p = {pp_d:.2e}')
print(f'Spearman(total_str, mean F_corr)         = {rho_s:+.3f}  p = {p_s:.2e}')
print(f'partial   (total_str, mean F_corr | dist)= {prho_s:+.3f}  p = {pp_s:.2e}')

fc_lim = max(abs(np.percentile(df_h.mean_fcorr, 1)), abs(np.percentile(df_h.mean_fcorr, 99)))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, x, lbl, rho, p_ in [
    (axes[0], df_h.total_deg, 'total degree (in + out)', rho_d, p_d),
    (axes[1], np.log10(df_h.total_str + 1), 'log10 total synapse strength + 1', rho_s, p_s),
]:
    sc = ax.scatter(x, df_h.mean_fcorr, c=df_h.mean_fcorr, cmap='RdBu_r',
                    vmin=-fc_lim, vmax=+fc_lim, s=20, alpha=0.85, linewidths=0.3, edgecolor='0.3')
    m, b = np.polyfit(x, df_h.mean_fcorr, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, m * xs + b, color='0.15', ls='--', lw=1.4)
    ax.set(xlabel=lbl, ylabel='mean $F_{corr}$ with cohort',
           title=f'Spearman ρ = {rho:+.3f}, p = {p_:.1e}')
    fig.colorbar(sc, ax=ax, shrink=0.85, label='mean $F_{corr}$')
plt.tight_layout(); plt.show()

H7_RESULT = {'test': 'H7 hub coupling',
             'spearman_deg': float(rho_d), 'p_deg': float(p_d),
             'partial_spearman_deg|dist': float(prho_d), 'p_partial_deg|dist': float(pp_d),
             'spearman_str': float(rho_s), 'p_str': float(p_s),
             'partial_spearman_str|dist': float(prho_s), 'p_partial_str|dist': float(pp_s)}


---
## §9 — H8: motifs vs a degree-preserving null

Motifs are the standard cortical-microcircuit signature (Milo et al. 2002, Song et al. 2005). We count:

- **2-node motifs:** unidirectional and bidirectional pairs; report the reciprocity ratio `bi / (uni + bi)`.
- **3-node motifs (transitive triangles):** A→B→C with A→C; the canonical excess pattern in cortex.

The null distribution preserves each node's in- and out-degree (`networkx.directed_configuration_model`), with parallel edges and self-loops removed. 200 randomisations; we report z-scores. The null histograms use a neutral mid-mako tone; the observed value is overlaid in the categorical "bi" red (`#E15759`) so the eye picks it out.

In [ ]:
# 9.0  H8 — motif z-scores under degree-preserving null
import networkx as nx

G = nx.DiGraph()
G.add_nodes_from(range(N))
nz = C.tocoo()
mask = nz.data > 0
G.add_edges_from(zip(nz.row[mask].tolist(), nz.col[mask].tolist()))
print(f'G: |V|={G.number_of_nodes()}, |E|={G.number_of_edges()}')

def count_motifs(g):
    A = nx.to_numpy_array(g, nodelist=list(range(g.number_of_nodes())), dtype=bool)
    np.fill_diagonal(A, False)
    rec = (A & A.T)
    n_bi  = int(rec.sum() // 2)
    n_uni = int((A & ~A.T).sum())
    Ai = A.astype(np.int32)
    n_trans = int(((Ai @ Ai) * Ai).sum())
    return n_bi, n_uni, n_trans

n_bi_obs, n_uni_obs, n_trans_obs = count_motifs(G)
recip_obs = n_bi_obs / max(n_bi_obs + n_uni_obs, 1)
print(f'observed:  bi={n_bi_obs:,}  uni={n_uni_obs:,}  reciprocity={recip_obs:.3f}')
print(f'observed transitive triangles: {n_trans_obs:,}')

n_null = 200
in_seq  = [d for _, d in G.in_degree(range(N))]
out_seq = [d for _, d in G.out_degree(range(N))]
null_bi, null_uni, null_trans = [], [], []
rng = np.random.default_rng(2)
for k in range(n_null):
    seed = int(rng.integers(1 << 31))
    Hm = nx.directed_configuration_model(in_seq, out_seq, seed=seed)
    H = nx.DiGraph()
    H.add_nodes_from(range(N))
    H.add_edges_from(Hm.edges())
    H.remove_edges_from(nx.selfloop_edges(H))
    nb, nu, nt = count_motifs(H)
    null_bi.append(nb); null_uni.append(nu); null_trans.append(nt)

null_bi = np.array(null_bi); null_uni = np.array(null_uni); null_trans = np.array(null_trans)

def zscore_obs(obs, null):
    return (obs - null.mean()) / max(null.std(ddof=1), 1e-9)

z_bi    = zscore_obs(n_bi_obs,    null_bi)
z_recip = zscore_obs(recip_obs,   null_bi / np.maximum(null_bi + null_uni, 1))
z_trans = zscore_obs(n_trans_obs, null_trans)
print(f'z(bidirectional count)        = {z_bi:+.2f}')
print(f'z(reciprocity ratio)          = {z_recip:+.2f}')
print(f'z(transitive triangles)       = {z_trans:+.2f}')

null_color = RAMP_SEQ(0.55)
obs_color  = PAL_CONN['bi']     # red highlight for observed value

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4))
for ax, obs, null, name in [
    (axes[0], n_bi_obs,    null_bi,    'bidirectional pairs'),
    (axes[1], null_bi / np.maximum(null_bi + null_uni, 1), None, 'reciprocity ratio'),
    (axes[2], n_trans_obs, null_trans, 'transitive triangles'),
]:
    if name == 'reciprocity ratio':
        ax.hist(obs, bins=30, color=null_color, edgecolor='white', alpha=0.85)
        ax.axvline(recip_obs, color=obs_color, lw=2, label=f'observed = {recip_obs:.3f}\nz = {z_recip:+.2f}')
    else:
        ax.hist(null, bins=30, color=null_color, edgecolor='white', alpha=0.85)
        ax.axvline(obs, color=obs_color, lw=2, label=f'observed = {obs:,}\nz = {zscore_obs(obs, null):+.2f}')
    ax.set_title(name); ax.legend()
fig.suptitle(f'H8 — motif counts vs degree-preserving null (configuration model, n_null={n_null})', y=1.02)
plt.tight_layout(); plt.show()

H8_RESULT = {'test': 'H8 motifs', 'z_bidirectional': float(z_bi),
             'z_reciprocity': float(z_recip), 'z_transitive': float(z_trans)}


---
## §10 — 93-neuron L4 sanity bookend

We re-run the H1 test on the homogeneous L4-V1 subset (`G_93`) using the cached `outputs/functional_network/F_correlation_matrix.npy`. The point of this section is to confirm that the headline direction is the same in a smaller, layer-pure slice — *not* to reach the same significance.

In [ ]:
# 10.0  93-neuron sanity check
F93        = np.load('outputs/functional_network/F_correlation_matrix.npy')
fc93_meta  = pd.read_csv('outputs/functional_network/functional_cohort.csv')
edges93    = pd.read_csv(EDGES_93)
nodes93    = pd.read_csv(NODES_93)

n93 = F93.shape[0]
id_to_idx_93 = {nid: i for i, nid in enumerate(fc93_meta.pt_root_id)}
A93 = np.zeros((n93, n93), dtype=bool)
for pre, post in zip(edges93.pre_neuron_id, edges93.post_neuron_id):
    if pre in id_to_idx_93 and post in id_to_idx_93:
        A93[id_to_idx_93[pre], id_to_idx_93[post]] = True

A93_und = A93 | A93.T
np.fill_diagonal(A93_und, False)
iu, ju = np.triu_indices(n93, k=1)
finite = np.isfinite(F93[iu, ju])
iu, ju = iu[finite], ju[finite]
fc93_pairs = F93[iu, ju]
conn93     = A93_und[iu, ju]

a, b = fc93_pairs[conn93], fc93_pairs[~conn93]
U_, p_ = mannwhitneyu(a, b, alternative='greater')
md_, ci_ = boot_mean_diff(a, b, n_boot=4000)
print(f'93-cohort  Δmean = {md_:+.4f}  95% CI [{ci_[0]:+.4f}, {ci_[1]:+.4f}]  p = {p_:.2e}  '
      f'(n_conn={conn93.sum()}, n_unconn={(~conn93).sum()})')

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(b, bins=40, density=True, histtype='step', lw=1.8, color=PAL_CONN['none'],
        label=f'unconnected (n={len(b)})')
ax.hist(a, bins=40, density=True, histtype='step', lw=1.8, color=PAL_CONN['uni'],
        label=f'connected   (n={len(a)})')
ax.axvline(0, color='0.4', lw=0.7, ls=':')
ax.set(xlabel='$F_{corr}$', ylabel='density',
       title=f'§10 — L4-V1 bookend (n=93): Δmean = {md_:+.4f}, p = {p_:.1e}')
ax.legend(); plt.tight_layout(); plt.show()

H10_RESULT = {'test': '93-neuron bookend', 'mean_diff': float(md_),
              'p_one_sided': float(p_), 'n_pairs': int(len(fc93_pairs))}


---
## §11 — Synthesis figure: 3D structural network coloured by `F_corr`

This is the money plot. We adapt the **last cell of `Dima/construct_matrices.ipynb`** *with its original palette intact*:

- **Edges** are taken from `C` (structural). For each edge we look up `F_corr[i, j]` (functional). Edges are **binned and coloured by `F_corr` along `coolwarm`** with symmetric ±99-th-percentile colour limits.
- **Neurons** are coloured by **brain area** with Dima's literal swatches — `V1` light green `#90EE90`, `RL` pale yellow `#FFFF99`, `AL` light blue `#87CEFA` — over a black background.
- **Bounding boxes** per area give 3D context.
- **Colorbar** uses `RdBu_r` (matches Dima's original).
- Saved to `outputs/figures/final_structure_function_3d.html` so the figure is shareable without rerunning the notebook.

We deliberately use the structural graph (edges from `C`) coloured by the functional similarity matrix `F_corr` — these are *not* mixed up: the geometry is the connectome, the colour conveys what each connected pair carries functionally.

In [ ]:
# 11.0  3D synthesis figure (plotly, Dima's exact palette)
import plotly.graph_objects as go

xyz = matched[['pt_position_x', 'pt_position_y', 'pt_position_z']].to_numpy()

A_und = (C + C.T).tocoo()
mask  = A_und.row < A_und.col
edges = np.stack([A_und.row[mask], A_und.col[mask]], axis=1)
edge_corr = F_corr[edges[:, 0], edges[:, 1]]
finite_e = np.isfinite(edge_corr)
edges, edge_corr = edges[finite_e], edge_corr[finite_e]
print(f'edges drawn: {len(edges):,}   F_corr range: [{edge_corr.min():+.3f}, {edge_corr.max():+.3f}]')

n_bins = 16
vlim   = max(abs(np.percentile(edge_corr, 1)), abs(np.percentile(edge_corr, 99)))
bins   = np.linspace(-vlim, +vlim, n_bins + 1)
cmap   = cm.coolwarm
norm   = Normalize(-vlim, +vlim)

traces = []
for b in range(n_bins):
    lo, hi = bins[b], bins[b + 1]
    sel = (edge_corr >= lo) & (edge_corr <= hi if b == n_bins - 1 else edge_corr < hi)
    if not sel.any():
        continue
    e   = edges[sel]
    pts = np.empty((len(e) * 3, 3))
    pts[0::3] = xyz[e[:, 0]]
    pts[1::3] = xyz[e[:, 1]]
    pts[2::3] = np.nan
    mid     = 0.5 * (lo + hi)
    color   = to_hex(cmap(norm(mid)))
    opacity = float(np.clip(abs(mid) / vlim, 0.1, 0.8))
    traces.append(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='lines',
        line=dict(color=color, width=1),
        opacity=opacity, hoverinfo='skip', showlegend=False, legendgroup='edges',
    ))

# Per-area neurons + translucent bounding boxes (Dima's literal swatches)
AREA_COLORS = {'V1': '#90EE90', 'RL': '#FFFF99', 'AL': '#87CEFA'}      # light green / yellow / blue

def cube_mesh(pts):
    mn, mx = pts.min(axis=0), pts.max(axis=0)
    X = [mn[0], mx[0], mx[0], mn[0], mn[0], mx[0], mx[0], mn[0]]
    Y = [mn[1], mn[1], mx[1], mx[1], mn[1], mn[1], mx[1], mx[1]]
    Z = [mn[2], mn[2], mn[2], mn[2], mx[2], mx[2], mx[2], mx[2]]
    i = [0, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2]
    j = [1, 2, 4, 1, 5, 6, 5, 2, 0, 1, 6, 3]
    k = [2, 3, 5, 5, 6, 7, 1, 1, 7, 4, 7, 6]
    return X, Y, Z, i, j, k

per_neuron_mean = np.nanmean(np.where(np.eye(N, dtype=bool), np.nan, F_corr), axis=1)
out_deg = np.asarray((C != 0).sum(axis=1)).ravel()
in_deg  = np.asarray((C != 0).sum(axis=0)).ravel()
out_str = np.asarray(C.sum(axis=1)).ravel()

for area in ['V1', 'RL', 'AL']:
    sel = (matched.brain_area == area).to_numpy()
    if not sel.any():
        continue
    pts = xyz[sel]
    color = AREA_COLORS[area]
    grp   = f'area_{area}'
    Xb, Yb, Zb, ii, jj, kk = cube_mesh(pts)
    traces.append(go.Mesh3d(
        x=Xb, y=Yb, z=Zb, i=ii, j=jj, k=kk,
        color=color, opacity=0.08, flatshading=True,
        hoverinfo='skip', showlegend=False, legendgroup=grp,
    ))
    hover = [
        f'idx={i}<br>{area} {l} {ct}<br>'
        f'mean F_corr={mfc:+.3f}<br>out_deg={od}, in_deg={idg}<br>out_str={os_:.0f}'
        for i, l, ct, mfc, od, idg, os_ in zip(
            matched.index[sel], matched.layer[sel], matched.cell_type[sel],
            per_neuron_mean[sel], out_deg[sel], in_deg[sel], out_str[sel],
        )
    ]
    traces.append(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
        marker=dict(size=2.5, color=color, line=dict(width=0)),
        name=f'{area}  ({sel.sum()})',
        text=hover, hoverinfo='text', legendgroup=grp, showlegend=True,
    ))

# Invisible trace just to draw the F_corr colorbar (RdBu_r — Dima's original)
traces.append(go.Scatter3d(
    x=[xyz[0, 0]], y=[xyz[0, 1]], z=[xyz[0, 2]], mode='markers',
    marker=dict(
        size=0.001, color=[0], cmin=-vlim, cmax=+vlim, colorscale='RdBu_r',
        colorbar=dict(title=dict(text='F_corr', font=dict(color='white')),
                      tickfont=dict(color='white'), x=1.02, len=0.7),
    ),
    hoverinfo='skip', showlegend=False,
))

fig = go.Figure(data=traces)
fig.update_layout(
    paper_bgcolor='black', plot_bgcolor='black', font=dict(color='white'),
    title=dict(
        text=(
            'Structure carries function — edges of the 906-neuron connectome coloured by Pearson signal correlation<br>'
            f'<sub>edges drawn: {len(edges):,}   |   '
            f'mean F_corr (connected) − mean F_corr (unconnected) = {H1_RESULT["mean_diff"]:+.4f}</sub>'
        ),
        font=dict(color='white'),
    ),
    legend=dict(
        font=dict(color='white'), bgcolor='rgba(0,0,0,0.5)',
        bordercolor='gray', borderwidth=1,
        itemclick='toggle', itemdoubleclick='toggleothers',
        x=0.01, y=0.99,
    ),
    scene=dict(
        bgcolor='black', aspectmode='data',
        xaxis=dict(title='x (µm)', backgroundcolor='black', color='white', gridcolor='dimgray'),
        yaxis=dict(title='y (µm)', backgroundcolor='black', color='white', gridcolor='dimgray'),
        zaxis=dict(title='z (µm)', backgroundcolor='black', color='white', gridcolor='dimgray'),
    ),
    width=1100, height=820, margin=dict(l=0, r=0, b=0, t=70),
)

OUT_HTML = OUT_FIG / 'final_structure_function_3d.html'
fig.write_html(str(OUT_HTML), include_plotlyjs='cdn', full_html=True)
print(f'saved: {OUT_HTML}')
fig.show()


---
## §12 — Summary

The table below collects effect sizes / statistics for every hypothesis test in the notebook. It's saved to `outputs/tables/like_to_like_summary.csv` for downstream use.

In [ ]:
# 12.0  Summary table
rows = [H1_RESULT, H2_RESULT, H3_RESULT, H4_MATCH, H4_LOGIT, H5_RESULT, H6_RESULT,
        H7_RESULT, H8_RESULT, H10_RESULT]
summary = pd.DataFrame(rows)
out_csv = OUT_TBL / 'like_to_like_summary.csv'
summary.to_csv(out_csv, index=False)
print(f'saved: {out_csv}')
summary


### §12.1 Interpretation

1. **Like-to-like.** H1 is the headline test: is `mean F_corr | connected > mean F_corr | unconnected`? Read Δmean and 95 % CI from the §2 cell — a positive Δ with a CI excluding 0 confirms the rule on this cohort.
2. **Distance is the dominant confound.** The shrinkage from raw Δmean (§2) to **distance-matched** Δmean (§5.1) and to the partial logistic coefficient on `f_corr` (§5.2) tells us how much of H1 was just "wired things are close, and close things fire alike". Whatever remains after the matched / partial regression tests is the signal-correlation rule itself.
3. **Composition adds little.** §6's joint logistic puts `f_corr`, `dist`, `same_area`, `same_layer`, `same_celltype` in the same model; if `f_corr` keeps a positive standardised β, like-to-like is a feature of the wiring beyond pure compositional alignment.
4. **Orientation is a coarser feature.** §7 reports both an `ori_sim` test on its own and a joint logistic — in the Wang et al. 2025 picture, signal correlation eats orientation similarity once both are in the model. We get to see whether our cohort agrees.
5. **Hubs and motifs.** §8 / §9 are bonus structure-only and structure-function bridge results; they don't change the H1 verdict but contextualise it.


### §13 Limitations (recap of `PLAN.md` §6)

1. **No bystander control.** MICrONS 2025 compared connected pairs to *axon–dendrite-apposed but unconnected* "bystanders". We only have soma positions, so our distance match is a coarser proxy.
2. **`F` is built from oracle stimuli only** (the intersection of condition hashes across 13 scans). Some response selectivity may not be expressed, so `F_corr` is plausibly an underestimate of true signal correlation.
3. **Synapse size is a proxy for strength**, not measured EPSC amplitude.
4. **Cohort spans V1+RL+AL across 13 scans.** Cross-scan `F_corr` is computed on stimuli, not co-recorded activity, so pure noise correlations cannot be tested at all — we are firmly in signal-correlation territory.
5. **`ax_clean` makes the out-side reliable** but the in-side mixes proofread targets with synapses from non-proofread presynaptic neurons. Bidirectional motif counts (§9) inherit this asymmetry.
